In [0]:
#pip install openpyxl

In [0]:
import pandas as pd
df_pricing_pd = pd.read_excel("/Workspace/Users/ranjan.mishra@cineplex.com/Offline Data/bf2025 cde.xlsx")
df_chargb = spark.createDataFrame(df_pricing_pd)


In [0]:
from pyspark.sql import SparkSession
def create_spark_session(app_name: str = "CineClubWrapped") -> SparkSession:

    return (
        SparkSession.builder
        .appName(app_name)
        # .config(...)  # cluster / warehouse configs if needed
        .getOrCreate()
    )

from pyspark.sql import functions as F

T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_sessions = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_session")
T_locations = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_films = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_film")
T_ticket = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_ticket_type")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_items = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")
T_genre = spark.table("cpx_dataanalytics_gold.customer360.film_composite_genres")
T_subscriber = spark.table("cpx_dataanalytics_gold.marketing.subscriber_profile_snapshot")
T_events = spark.table("cpx_dataanalytics_gold.marketing.subscription_events")

In [0]:
df_upgraded_new= (
    T_subscriber.filter(F.col("snapshot_date_id") == 20251216)
    .join(T_events, "cc_user_profile_id", how="inner")
    .filter(
        (F.col("reporting_datetime").between("2025-11-28", "2025-12-01")) &
        (F.col("changed") == True) &
        (F.col("changed_from").like("%monthly%")) &
        (F.col("changed_to").like("%annual%"))
    )
)

df_upgraded_new.count()


In [0]:
df_members_renewed_new = (
    T_subscriber.filter(F.col("snapshot_date_id") == 20251216)
    .join(T_events, on="cc_user_profile_id", how="inner")
    .filter(
        (F.col("reporting_datetime").between("2025-11-28", "2025-12-01")) &
        (F.col("event_type") == "subscription_renewed") &
        (F.col("status") == "active") &
        (T_events.plan_id.like("%annual%")) &
        (F.col("total") > 90)
    )
)

In [0]:
df_members_renewed_new.count()

In [0]:
df_details = (
    T_subscriber.filter(F.col("snapshot_date_id") == 20251216)
    .filter(F.to_date(T_subscriber.activated_at_datetime).between("2025-11-28", "2025-12-01"))
    .filter(F.col("plan_id") == "annual1")
    .select('cde_id',T_subscriber.activated_at_datetime)
    .distinct()
)

df_first_activation_new = (
    T_subscriber
    .groupBy("cde_id")
    .agg(
        F.min(F.to_date(T_subscriber.activated_at_datetime))
    )
    .withColumnRenamed("min(to_date(activated_at_datetime))","first_activation")
)


df_details.count()

In [0]:
%sql
select cde_id
      ,min(to_date(activated_at_datetime)) as first_activation
from cpx_dataanalytics_gold.marketing.subscriber_profile_snapshot
where cde_id in (
select distinct cde_id
from cpx_dataanalytics_gold.marketing.subscriber_profile_snapshot
where snapshot_date_id = 20251206
and to_date(activated_at_datetime) between '2025-11-28' and '2025-12-01'
and plan_id = 'annual1'
)
group by cde_id

In [0]:
df_details = (df_details
              .join(df_first_activation_new, on="cde_id", how="inner")
              )

In [0]:
df_returning = (
    df_details
    .filter(~F.col("first_activation").between("2025-11-28", "2025-12-01"))
)

df_returning.count()

In [0]:
df_eligible_cdes = (
    T_subscriber
    .filter(F.col("snapshot_date_id") == 20251202)
    .filter(F.to_date("activated_at_datetime").between("2025-11-28", "2025-12-01"))
    .filter(F.col("plan_id") == "annual1")
    .select("cde_id")
    .distinct()
)

df_first_activation = (
    T_subscriber
    .join(df_eligible_cdes, on="cde_id", how="inner")
    .groupBy("cde_id")
    .agg(
        F.min(F.to_date("activated_at_datetime")).alias("first_activation")
    )
)

df_returning = (
    df_first_activation
    .filter(~F.col("first_activation").between("2025-11-28", "2025-12-01"))
)

df_net_new = (
    df_first_activation
    .filter(F.col("first_activation").between("2025-11-28", "2025-12-01"))
)

df_members_upgraded = (
    T_events
    .filter(
        (F.col("reporting_datetime").between("2025-11-28", "2025-12-01")) &
        (F.col("changed") == True) &
        (F.col("changed_from").like("%monthly%")) &
        (F.col("changed_to").like("%annual%"))
    )
)

df_members_renewed = (
    T_events
    .filter(
        (F.col("reporting_datetime").between("2025-11-28", "2025-12-01")) &
        (F.col("event_type") == "subscription_renewed") &
        (F.col("status") == "active") &
        (F.col("plan_id").like("%annual%")) &
        (F.col("total") > 90)
    )
)

df_members_renewed.count()

In [0]:
df_net_new_cdes = df_net_new.select("cde_id")

df_returning_cdes = df_returning.select("cde_id")

matched_members_1 = (
    T_subscriber
    .join(df_members_upgraded, "cc_user_profile_id", "inner")
    .select("CDE_ID")
    .distinct()
)

matched_members_2 = (
    T_subscriber
    .join(df_members_renewed, "cc_user_profile_id", "inner")
    .select("CDE_ID")
    .distinct()
)

all_members = (
    df_net_new_cdes
    .union(df_returning_cdes)
    .union(matched_members_1)
    .union(matched_members_2)
    .distinct()
)


In [0]:
all_members = df_chargb

In [0]:
all_members.distinct().count()

In [0]:
from pyspark.sql import functions as F

T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_locations = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_items = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")

T_master_trans = (T_trans
    .join(T_locations, T_trans.LocationId == T_locations.LocationID, "left")
    .join(T_items, T_trans.ItemSkey == T_items.itemskey, "left")
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .filter(
        (T_locations.LC_Location_Type_Description == "Cineplex Theatre")
        & (F.col("STOREFLAG") == "No")
        & (F.col("RecRoomFLAG") == "No")
        & (T_trans.CDE_ID.isNotNull())
        & (F.col("datefull").between("2025-01-01", "2025-12-04"))
        & ((F.col("TicketFLAG") == "Yes") | (F.col("ConcessionFLAG") == "Yes"))
    )
    .withColumnRenamed("datefull", "VisitDate")
)

df_all_member_trans = all_members.join(T_master_trans, "CDE_ID", "inner")

In [0]:
df_Visits = (
    df_all_member_trans
    .select(
        "CDE_ID",
        "VisitDate",
        T_trans.LocationId,
        "TicketFLAG",
        "ConcessionFLAG",
        "Item_Description",
        "Quantity"
    )
    # Flag popcorn presence at transaction level
    .withColumn(
        "PopcornFLAG",
        F.when(F.col("Item_Description").like("%POPCORN%"), 1).otherwise(0)
    )
    # Collapse to one row per visit
    .groupBy(
        "CDE_ID",
        "VisitDate",
        T_trans.LocationId,

    )
    .agg(
        F.sum("Quantity").alias("Net_Quantity"),        # refund-safe quantity
        F.max("PopcornFLAG").alias("Popcorn_Visit")     # binary popcorn incidence per visit
    )
)



df_total_visits = (
    df_Visits
    .withColumn("Month", F.date_format(F.col("VisitDate"), "yyyy-MM"))
    .groupBy("CDE_ID", "Month")
    .agg(
        F.count("*").alias("Total_Visits"),
        F.sum("Popcorn_Visit").alias("Popcorn_Visits")
    )
)


In [0]:
df_p12m = df_total_visits.groupby("CDE_ID").agg(
    F.sum("Total_Visits").alias("Total_Visits"),
    F.sum("Popcorn_Visits").alias("Popcorn_Visits"))

In [0]:
df_p12m.display()



In [0]:
df_total_visits.display()

How Many Left the Program

In [0]:
df_joined_last_year = (T_subscriber
    .filter(F.col("snapshot_date_id") == 20251216)
    .filter(F.to_date(T_subscriber.activated_at_datetime).between("2025-11-28", "2025-12-31"))
    .filter(F.col("plan_id") == "annual1")
    .select("cde_id","activated_at_datetime", "renewal_status")
    .distinct()
)

In [0]:
df_joined_last_year.count()

